In [1]:
'''
looking at random forest outputs
'''

'\nlooking at random forest outputs\n'

In [2]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import precision_recall as pr
import gc  # garbage collector
import review_findings_functions as rff
import os
import importlib
import implantation_functions as imp
# import corner
# from astropy.wcs import WCS
# import shutil
# from astroquery.gaia import Gaia
# from astropy.coordinates import SkyCoord
# import astropy.units as u
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gist_gray"

Matplotlib is building the font cache; this may take a moment.


In [38]:
predictions = pd.read_csv('/lustre/lrspec/ZTF_data/random_forest_training/predictions_updated.csv', names=['filename', 'rf'])

In [39]:
predictions

,filename,rf
0,lightcurve_ztf_20221020138241_000686_zi_c11_x_...,0
1,lightcurve_ztf_20221020148032_000686_zg_c13_x_...,0
2,lightcurve_ztf_20221020157697_000640_zi_c01_x_...,0
3,lightcurve_ztf_20221020138241_000686_zi_c04_x_...,0
4,lightcurve_ztf_20221020163819_000640_zg_c10_x_...,0
...,...,...
723141,lightcurve_ztf_20221020138241_000686_zi_c13_x_...,0
723142,lightcurve_ztf_20221020163819_000640_zg_c12_x_...,0
723143,lightcurve_ztf_20221020148032_000686_zg_c07_x_...,0
723144,lightcurve_ztf_20221020163819_000640_zg_c16_x_...,0


In [35]:
predictions

,filename,rf
0,lightcurve_ztf_20221020138241_000686_zi_c11_x_...,1
1,lightcurve_ztf_20221020148032_000686_zg_c13_x_...,0
2,lightcurve_ztf_20221020157697_000640_zi_c01_x_...,0
3,lightcurve_ztf_20221020138241_000686_zi_c04_x_...,0
4,lightcurve_ztf_20221020163819_000640_zg_c10_x_...,0
...,...,...
723141,lightcurve_ztf_20221020138241_000686_zi_c13_x_...,0
723142,lightcurve_ztf_20221020163819_000640_zg_c12_x_...,0
723143,lightcurve_ztf_20221020148032_000686_zg_c07_x_...,1
723144,lightcurve_ztf_20221020163819_000640_zg_c16_x_...,0


In [34]:
#i think 0 is good, 1 is bad
predictions.loc[0, 'filename']

'lightcurve_ztf_20221020138241_000686_zi_c11_x_0023_4_1499_1443.csv'

In [5]:
#this is not the most recent but just to test
concat = pd.read_csv('/lustre/lrspec/ZTF_data/science_extractions/1020april26/average/concat')

In [25]:
matching_row

,filename,rf
409200,lightcurve_ztf_20221020138241_000686_zi_c03_x_...,0


In [30]:
concat

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,filename,quad,type,row_from_nn,col_from_nn,detection_size,closest_x,...,noise_local_dimming,noise_global_dimming,local_average_outside_dimming,height_dimming,FWHM_dimming,signal_local_dimming,central_point_dimming,quadrant,num_in_file_in_star,random_forest_result
0,0,0,0,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,686.500000,907.000000,2,919.0,...,216.645045,171.717937,6334.883496,-263.138867,2.0,20.657016,703.0,4,6.0,0.0
1,1,1,3,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,1163.312500,908.625000,16,919.0,...,157.363985,173.840620,6286.510950,-252.650110,3.0,25.764495,1168.0,4,6.0,0.0
2,2,2,4,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,2080.173913,909.000000,23,919.0,...,136.800273,170.488499,6321.075952,-37.555444,1.0,6.128250,2082.0,4,6.0,0.0
3,3,3,6,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,2457.625000,909.375000,8,919.0,...,179.325276,170.457841,6331.314060,-165.742283,2.0,16.219192,2461.0,4,6.0,0.0
4,4,4,7,ztf_20221020141806_000686_zi_c11_x_0017.,4,both,2561.500000,908.000000,6,919.0,...,166.683383,170.003755,6372.875215,-509.109590,2.0,31.840958,2578.0,4,6.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347419,347419,49,77,ztf_20221020157697_000640_zi_c05_x_0024.,3,neg,824.500000,2703.000000,2,2714.0,...,39.331224,37.040312,1004.303713,-121.429384,2.0,14.854293,833.0,3,7.0,NaN
347420,347420,50,34,ztf_20221020157697_000640_zi_c05_x_0024.,3,both,1247.214286,2904.642857,13,2913.0,...,15.551740,16.063058,338.402133,-60.982577,4.0,14.286691,1253.0,3,2.0,NaN
347421,347421,51,39,ztf_20221020157697_000640_zi_c05_x_0024.,3,both,1350.156250,2904.000000,21,2913.0,...,18.688348,16.210222,334.067805,-33.243006,1.0,5.765675,1366.0,3,2.0,NaN
347422,347422,0,0,ztf_20221020294271_000449_zr_c04_x_0003.,1,pos,830.857143,1381.857143,14,1392.0,...,204.000038,229.755277,11174.133823,-432.334995,2.0,27.397520,830.0,1,1.0,NaN


In [37]:
concat

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,filename,quad,type,row_from_nn,col_from_nn,detection_size,closest_x,...,noise_local_dimming,noise_global_dimming,local_average_outside_dimming,height_dimming,FWHM_dimming,signal_local_dimming,central_point_dimming,quadrant,num_in_file_in_star,random_forest_result
0,0,0,0,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,686.500000,907.000000,2,919.0,...,216.645045,171.717937,6334.883496,-263.138867,2.0,20.657016,703.0,4,6.0,0.0
1,1,1,3,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,1163.312500,908.625000,16,919.0,...,157.363985,173.840620,6286.510950,-252.650110,3.0,25.764495,1168.0,4,6.0,0.0
2,2,2,4,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,2080.173913,909.000000,23,919.0,...,136.800273,170.488499,6321.075952,-37.555444,1.0,6.128250,2082.0,4,6.0,0.0
3,3,3,6,ztf_20221020141806_000686_zi_c11_x_0017.,4,pos,2457.625000,909.375000,8,919.0,...,179.325276,170.457841,6331.314060,-165.742283,2.0,16.219192,2461.0,4,6.0,0.0
4,4,4,7,ztf_20221020141806_000686_zi_c11_x_0017.,4,both,2561.500000,908.000000,6,919.0,...,166.683383,170.003755,6372.875215,-509.109590,2.0,31.840958,2578.0,4,6.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347419,347419,49,77,ztf_20221020157697_000640_zi_c05_x_0024.,3,neg,824.500000,2703.000000,2,2714.0,...,39.331224,37.040312,1004.303713,-121.429384,2.0,14.854293,833.0,3,7.0,NaN
347420,347420,50,34,ztf_20221020157697_000640_zi_c05_x_0024.,3,both,1247.214286,2904.642857,13,2913.0,...,15.551740,16.063058,338.402133,-60.982577,4.0,14.286691,1253.0,3,2.0,NaN
347421,347421,51,39,ztf_20221020157697_000640_zi_c05_x_0024.,3,both,1350.156250,2904.000000,21,2913.0,...,18.688348,16.210222,334.067805,-33.243006,1.0,5.765675,1366.0,3,2.0,NaN
347422,347422,0,0,ztf_20221020294271_000449_zr_c04_x_0003.,1,pos,830.857143,1381.857143,14,1392.0,...,204.000038,229.755277,11174.133823,-432.334995,2.0,27.397520,830.0,1,1.0,NaN


In [36]:
for i, df_row in concat.iterrows():
    lc_filename = 'lightcurve_'+df_row['filename'][:-1]+'_'+str(df_row['quadrant'])+'_'+f'{int(df_row["closest_x"]):04}'+'_'+f'{int(df_row["closest_y"]):04}'+'.csv'
    matching_row = predictions[predictions['filename']==lc_filename]
    # print(lc_filename)
    # print(matching_row['rf'].values[0])
    if len(matching_row)==1:
        concat.loc[i, 'random_forest_result']=matching_row['rf'].values[0]
        print('found it')
    elif len(matching_row)==0:
        # concat.loc[i, 'random_forest_result']=None #actually can just leave this as a NaN
        print('didnt find it')
    else:
        print(matching_row)
        print('something strange is happening...')
        print(len(matching_row))
    # break

found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it
found it


KeyboardInterrupt: 